# FleetMind-LLM
## Phase 1 — Logistics Instruction Dataset

### 1. 阶段目标

基于公开物流运营数据，构建用于 Qwen3-0.6B
监督微调与评测的中文指令数据集。

首版任务：
- Task A：物流业务信息提取，输出结构化 JSON。
- Task B：物流风险分类，标签与判断规则在数据检查后确定。

Task C：物流经营推理，作为后续拓展任务。
本阶段不训练模型。

### 2. 数据来源

- 数据集：Logistics Operations Database
- 发布者：Yogape Rodriguez
- 数据性质：国外货车运输业务背景的合成数据
- 数据地址：https://huggingface.co/datasets/yogape/logistics-operations

第一批检查文件：
- loads.csv：运单及收入信息。
- trips.csv：运输任务、车辆分配及运输表现。
- routes.csv：起终点及路线信息。
- delivery_events.csv：计划与实际交付时间等。

费用和其他风险任务所需的数据表，后续按需加入。

### 3. 数据处理原则

- 保留原始文件，通过代码生成处理后的数据。
- 先检查数据，再确定字段与标签。
- 中文表达保留原数据的地点、币种及业务含义。
- 不编造缺失信息、异常原因或完整成本。
- 同一业务记录衍生的样本放入同一个数据集合。
- 测试集独立保留，不用于训练或选择参数。

### 4. 计划产出

- train.jsonl：训练集。
- val.jsonl：验证集。
- test.jsonl：测试集。
- 字段与标签定义、数据来源说明及简要统计。

数据采用 Chat Messages 格式。
样本数量与划分比例在数据检查后确定。

## 1. 原始数据读取与初步检查

读取第一批四张业务表，检查行列数、字段名称和前几条样本。
此步骤保留全部原始字段，不删除或修改记录。

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import pandas as pd
from google.colab import files
from IPython.display import display

# 1.1 上传 ZIP 压缩包
uploaded = files.upload()
zip_files = [
    Path(name) for name in uploaded
    if name.lower().endswith(".zip")
]

if len(zip_files) != 1:
    raise ValueError("请只上传一个 ZIP 压缩包。")

# 1.2 从压缩包中查找四个 CSV
FILE_NAMES = [
    "loads.csv",
    "trips.csv",
    "routes.csv",
    "delivery_events.csv",
]

raw_tables = {}

with ZipFile(zip_files[0]) as archive:
    for filename in FILE_NAMES:
        matches = [
            name for name in archive.namelist()
            if Path(name).name == filename
            and "__MACOSX" not in Path(name).parts
        ]

        if len(matches) != 1:
            raise ValueError(
                f"{filename} 应有且只有一份，实际找到 {len(matches)} 份。"
            )

        with archive.open(matches[0]) as csv_file:
            raw_tables[Path(filename).stem] = pd.read_csv(csv_file)

# 1.3 汇总实际规模
summary = pd.DataFrame([
    {
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
    }
    for name, df in raw_tables.items()
])

display(summary)

# 1.4 查看字段和前三条记录
for name, df in raw_tables.items():
    print(f"\n{name}")
    print("字段：", df.columns.tolist())
    display(df.head(3))

## 2. 数据质量检查

检查主键缺失与重复、字段缺失、日期解析及事件类型。
本步骤只检查，不修改或删除原始记录。

In [ ]:
# 2.1 检查主键和整行重复
PRIMARY_KEYS = {
    "loads": "load_id",
    "trips": "trip_id",
    "routes": "route_id",
    "delivery_events": "event_id",
}

quality_summary = []

for name, df in raw_tables.items():
    key = PRIMARY_KEYS[name]

    quality_summary.append({
        "table": name,
        "rows": len(df),
        "missing_primary_key": int(df[key].isna().sum()),
        "duplicate_primary_key": int(
            df.loc[df[key].notna(), key].duplicated().sum()
        ),
        "duplicate_rows": int(df.duplicated().sum()),
    })

print("主键与重复检查")
display(pd.DataFrame(quality_summary))


# 2.2 只列出存在缺失的字段
missing_records = []

for name, df in raw_tables.items():
    for column, count in df.isna().sum().items():
        if count > 0:
            missing_records.append({
                "table": name,
                "column": column,
                "missing_count": int(count),
                "missing_pct": round(count / len(df) * 100, 2),
            })

print("\n缺失值检查")
if missing_records:
    display(pd.DataFrame(missing_records))
else:
    print("未发现缺失值")


# 2.3 检查日期是否能解析，不覆盖原始列
DATE_COLUMNS = {
    "loads": ["load_date"],
    "trips": ["dispatch_date"],
    "delivery_events": ["scheduled_datetime", "actual_datetime"],
}

date_checks = []

for name, columns in DATE_COLUMNS.items():
    for column in columns:
        original = raw_tables[name][column]
        parsed = pd.to_datetime(
            original, errors="coerce", format="mixed"
        )

        date_checks.append({
            "table": name,
            "column": column,
            "missing_original": int(original.isna().sum()),
            "parse_failures": int(
                (original.notna() & parsed.isna()).sum()
            ),
            "earliest": parsed.min(),
            "latest": parsed.max(),
        })

print("\n日期解析检查")
display(pd.DataFrame(date_checks))


# 2.4 查看事件类型及数量
print("\n事件类型")
print(
    raw_tables["delivery_events"]["event_type"]
    .value_counts(dropna=False)
    .to_string()
)

  **检查结果正常，可以继续。**
- 四张表的主键均无缺失、无重复，也没有整行重复。
- 只有司机、车辆和挂车编号存在约 2% 缺失。先保留，后续提取时可输出 null。
- 日期全部解析成功。跨年运输可能延续到 2025 年，不能仅因此删除。
- Pickup、`Delivery` 各 85,410 条；还需要确认是否每趟运输都恰好对应各一条。

## 3. 表间关联检查

检查关联编号是否存在、运单与运输任务是否一一对应，
以及每个运输任务是否包含一条提货事件和一条交付事件。
本步骤只检查，不合并生成最终业务数据。

In [ ]:
loads = raw_tables["loads"]
trips = raw_tables["trips"]
routes = raw_tables["routes"]
events = raw_tables["delivery_events"]

# 3.1 检查关联编号是否存在
RELATIONS = [
    ("loads", "route_id", "routes", "route_id"),
    ("trips", "load_id", "loads", "load_id"),
    ("delivery_events", "load_id", "loads", "load_id"),
    ("delivery_events", "trip_id", "trips", "trip_id"),
]

relation_checks = []

for child_name, child_key, parent_name, parent_key in RELATIONS:
    values = raw_tables[child_name][child_key]
    parent_values = raw_tables[parent_name][parent_key]

    relation_checks.append({
        "relation": f"{child_name} → {parent_name}",
        "key": child_key,
        "missing_key": int(values.isna().sum()),
        "unmatched_rows": int(
            (values.notna() & ~values.isin(parent_values)).sum()
        ),
    })

print("关联编号检查")
display(pd.DataFrame(relation_checks))


# 3.2 检查每个运单对应多少个运输任务
trips_per_load = (
    trips["load_id"].value_counts()
    .reindex(loads["load_id"], fill_value=0)
)

print("\n运单与运输任务对应检查")
display(pd.DataFrame([{
    "loads_without_trip": int((trips_per_load == 0).sum()),
    "loads_with_one_trip": int((trips_per_load == 1).sum()),
    "loads_with_multiple_trips": int((trips_per_load > 1).sum()),
}]))


# 3.3 检查每趟运输的事件数量
event_counts = pd.crosstab(events["trip_id"], events["event_type"])
event_counts = event_counts.reindex(
    index=trips["trip_id"],
    columns=["Pickup", "Delivery"],
    fill_value=0,
)

valid_event_counts = event_counts.eq(1).all(axis=1)

print("\n运输任务与事件数量检查")
display(pd.DataFrame([{
    "trips_checked": len(event_counts),
    "trips_with_one_pickup_and_one_delivery": int(
        valid_event_counts.sum()
    ),
    "trips_with_abnormal_event_counts": int(
        (~valid_event_counts).sum()
    ),
}]))


# 3.4 检查事件中的运单编号是否与对应运输任务一致
event_trip_check = events[["event_id", "trip_id", "load_id"]].merge(
    trips[["trip_id", "load_id"]],
    on="trip_id",
    how="left",
    suffixes=("_event", "_trip"),
    validate="many_to_one",
    indicator=True,
)

matched = event_trip_check["_merge"].eq("both")
mismatch = (
    matched
    & event_trip_check["load_id_event"].ne(
        event_trip_check["load_id_trip"]
    )
)

print("\n事件与运输任务的运单编号一致性检查")
print("匹配成功但运单编号不一致的事件数:", int(mismatch.sum()))

**关联检查全部通过。**
85,410 个运单均对应一趟运输，每趟各有一条提货和交付事件，关联编号也一致。

## 4. 合并运输业务记录

以运单为基础，关联运输任务、路线、提货及交付事件。
合并后每行代表一趟运输。

提货和交付字段分别使用 pickup_、delivery_ 前缀。
本步骤不生成风险标签，不计算完整利润。

In [ ]:
# 4.1 合并运单、运输任务和路线
business_records = (
    loads
    .merge(
        trips,
        on="load_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        routes,
        on="route_id",
        how="left",
        validate="many_to_one",
    )
)

# 4.2 分别关联提货和交付事件，避免一趟运输变成两行
EVENT_KEYS = ["load_id", "trip_id"]
EVENT_FIELDS = [
    "event_id",
    "scheduled_datetime",
    "actual_datetime",
    "detention_minutes",
    "on_time_flag",
    "location_city",
    "location_state",
]

for event_type, prefix in [
    ("Pickup", "pickup"),
    ("Delivery", "delivery"),
]:
    event_table = events.loc[
        events["event_type"].eq(event_type),
        EVENT_KEYS + EVENT_FIELDS,
    ].copy()

    event_table = event_table.rename(columns={
        column: f"{prefix}_{column}"
        for column in EVENT_FIELDS
    })

    business_records = business_records.merge(
        event_table,
        on=EVENT_KEYS,
        how="left",
        validate="one_to_one",
    )

# 4.3 验证合并没有丢失、扩增或重复业务记录
assert len(business_records) == len(loads), "合并前后行数不一致"
assert business_records["load_id"].is_unique, "运单编号重复"
assert business_records["trip_id"].is_unique, "运输任务编号重复"
assert business_records[
    ["pickup_event_id", "delivery_event_id"]
].notna().all().all(), "存在未关联的事件"

print("合并成功")
print("业务记录数:", len(business_records))
print("字段数:", business_records.shape[1])

# 4.4 展示关键业务字段
PREVIEW_COLUMNS = [
    "load_id",
    "trip_id",
    "truck_id",
    "origin_city",
    "destination_city",
    "revenue",
    "fuel_gallons_used",
    "pickup_scheduled_datetime",
    "pickup_actual_datetime",
    "delivery_scheduled_datetime",
    "delivery_actual_datetime",
]

display(business_records[PREVIEW_COLUMNS].head(3))

## 5. 业务数值与时间逻辑检查

检查数值有效性、提货与交付的时间顺序，以及事件地点与路线是否一致。
计算实际与计划交付时间的差值，用于了解延误分布。

检查结果只用于识别待核查记录，不自动删除数据。
延误差值为正表示晚到，为负表示提前到达。

In [ ]:
import numpy as np

# 5.1 创建副本，保留前一步的业务表
checked_records = business_records.copy()
checks = []

def record_check(name, mask):
    checks.append({
        "check": name,
        "flagged_rows": int(mask.sum()),
        "flagged_pct": round(mask.mean() * 100, 2),
    })


# 5.2 检查数值：严格大于零或允许为零
POSITIVE_COLUMNS = [
    "weight_lbs", "pieces",
    "actual_distance_miles", "actual_duration_hours",
    "fuel_gallons_used", "average_mpg",
    "typical_distance_miles", "typical_transit_days",
]

NONNEGATIVE_COLUMNS = [
    "revenue", "fuel_surcharge", "accessorial_charges",
    "idle_time_hours",
    "pickup_detention_minutes", "delivery_detention_minutes",
]

for column in POSITIVE_COLUMNS + NONNEGATIVE_COLUMNS:
    values = pd.to_numeric(checked_records[column], errors="coerce")
    finite = pd.Series(np.isfinite(values), index=values.index)

    record_check(f"{column}: 缺失或非有效数值", ~finite)

    invalid_range = (
        values.le(0) if column in POSITIVE_COLUMNS else values.lt(0)
    )
    record_check(f"{column}: 超出预期范围", finite & invalid_range)

    checked_records[column] = values


# 5.3 解析时间并检查先后顺序
TIME_COLUMNS = [
    "pickup_scheduled_datetime",
    "pickup_actual_datetime",
    "delivery_scheduled_datetime",
    "delivery_actual_datetime",
]

for column in TIME_COLUMNS:
    checked_records[column] = pd.to_datetime(
        checked_records[column],
        errors="coerce",
        format="mixed",
    )
    record_check(f"{column}: 缺失或解析失败",
                 checked_records[column].isna())

record_check(
    "实际交付早于实际提货",
    checked_records["delivery_actual_datetime"]
    < checked_records["pickup_actual_datetime"],
)

record_check(
    "计划交付早于计划提货",
    checked_records["delivery_scheduled_datetime"]
    < checked_records["pickup_scheduled_datetime"],
)


# 5.4 检查事件地点是否与路线一致
for event_column, route_column in [
    ("pickup_location_city", "origin_city"),
    ("pickup_location_state", "origin_state"),
    ("delivery_location_city", "destination_city"),
    ("delivery_location_state", "destination_state"),
]:
    record_check(
        f"{event_column}: 与路线不一致",
        checked_records[event_column].fillna("").astype(str).str.strip()
        != checked_records[route_column].fillna("").astype(str).str.strip(),
    )

print("业务逻辑检查")
display(pd.DataFrame(checks))


# 5.5 计算交付时间差，暂不生成风险标签
checked_records["delivery_time_diff_minutes"] = (
    checked_records["delivery_actual_datetime"]
    - checked_records["delivery_scheduled_datetime"]
).dt.total_seconds() / 60

print("\n实际交付时间减去计划时间（分钟）")
display(
    checked_records["delivery_time_diff_minutes"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99])
    .to_frame("minutes")
)

print(
    "晚于计划交付的记录数:",
    int(checked_records["delivery_time_diff_minutes"].gt(0).sum())
)

## 6. 轻量清洗与异常记录保留

排除以下记录：
1. 实际交付时间早于实际提货时间。
2. 计划交付时间早于计划提货时间。

两类异常取并集，避免重复计数。
异常记录单独保留，不直接修改或交换原始时间。

保留提前交付、延误及司机/车辆/挂车编号缺失的记录。
本步骤不生成风险标签。

In [ ]:
# 6.1 标记时间顺序异常
time_flags = pd.DataFrame({
    "actual_time_reversed": (
        checked_records["delivery_actual_datetime"]
        < checked_records["pickup_actual_datetime"]
    ),
    "scheduled_time_reversed": (
        checked_records["delivery_scheduled_datetime"]
        < checked_records["pickup_scheduled_datetime"]
    ),
})

invalid_time = time_flags.any(axis=1)

# 6.2 分开保留异常记录与可用记录，不修改原始表
excluded_records = checked_records.loc[invalid_time].copy()
excluded_records = excluded_records.join(
    time_flags.loc[invalid_time]
)

clean_records = checked_records.loc[~invalid_time].copy()

# 6.3 检查数量与唯一性
assert len(clean_records) + len(excluded_records) == len(checked_records)
assert clean_records["load_id"].is_unique
assert clean_records["trip_id"].is_unique

cleaning_summary = pd.DataFrame([{
    "before_rows": len(checked_records),
    "both_time_issues": int(time_flags.all(axis=1).sum()),
    "excluded_rows": len(excluded_records),
    "retained_rows": len(clean_records),
    "retained_pct": round(
        len(clean_records) / len(checked_records) * 100, 2
    ),
}])

print("清洗结果")
display(cleaning_summary)

print("\n保留数据中的交付时间差（分钟）")
display(
    clean_records["delivery_time_diff_minutes"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99])
    .to_frame("minutes")
)

## 7. 首批任务定义与标签生成

### 7.1 Task A：物流信息提取
输出字段：
load_id、truck_id、origin_city、destination_city、
revenue_usd、fuel_used_gallons、scheduled_delivery、actual_delivery。

仅提取输入中提供的信息；缺失信息输出 null。
保留原始币种、地点和计量单位。

### 7.2 Task B：交付延误分类
- on_time：实际交付时间不晚于计划时间。
- delayed：实际交付时间晚于计划时间。

标签依据原始时间计算，不使用原数据的 on_time_flag。
本任务评估已发生记录的延误识别与规则遵循能力，
不代表未来延误预测能力。

构建模型输入时，不提供派生时间差或答案标签。
费用及经营风险任务后续扩展。

In [ ]:
# 7.4 根据项目规则生成标签，保留清洗结果不变
task_records = clean_records.copy()

required_times = [
    "delivery_scheduled_datetime",
    "delivery_actual_datetime",
]
assert task_records[required_times].notna().all().all()

task_records["delay_label"] = np.where(
    task_records["delivery_actual_datetime"]
    > task_records["delivery_scheduled_datetime"],
    "delayed",
    "on_time",
)

label_summary = (
    task_records["delay_label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="count")
)

label_summary["percentage"] = (
    label_summary["count"] / len(task_records) * 100
).round(2)

print("交付延误标签分布")
display(label_summary)

## 8. 数据集划分

按运单划分：
- Training：约 80%。
- Validation：约 10%。
- Test：约 10%。

使用随机种子 42，按 delay_label 分层，保持标签比例接近。

先划分原始业务记录，再构建指令样本。
同一运单生成的 Task A、Task B 及不同表达必须属于同一个集合。

测试集保留用于最终评测，不用于调整提示词或训练参数。

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

# 8.2.1 确认一行对应一个唯一运单和运输任务
assert task_records["load_id"].is_unique
assert task_records["trip_id"].is_unique

# 8.2.2 先划分 80% 训练集、20% 临时集合
train_records, holdout_records = train_test_split(
    task_records,
    test_size=0.20,
    random_state=SEED,
    stratify=task_records["delay_label"],
)

# 8.2.3 将临时集合均分为验证集和测试集
val_records, test_records = train_test_split(
    holdout_records,
    test_size=0.50,
    random_state=SEED,
    stratify=holdout_records["delay_label"],
)

record_splits = {
    "train": train_records.copy().reset_index(drop=True),
    "validation": val_records.copy().reset_index(drop=True),
    "test": test_records.copy().reset_index(drop=True),
}

# 8.2.4 验证运单和运输任务没有跨集合重叠
for key in ["load_id", "trip_id"]:
    id_sets = [
        set(df[key]) for df in record_splits.values()
    ]
    assert id_sets[0].isdisjoint(id_sets[1])
    assert id_sets[0].isdisjoint(id_sets[2])
    assert id_sets[1].isdisjoint(id_sets[2])

assert sum(len(df) for df in record_splits.values()) == len(task_records)

# 8.2.5 汇总规模和标签比例
split_summary = pd.DataFrame([
    {
        "split": name,
        "records": len(df),
        "percentage": round(len(df) / len(task_records) * 100, 2),
        "delayed_pct": round(df["delay_label"].eq("delayed").mean() * 100, 2),
        "on_time_pct": round(df["delay_label"].eq("on_time").mean() * 100, 2),
    }
    for name, df in record_splits.items()
])

display(split_summary)
print("检查通过：各集合运单与运输任务均无重叠")

## 9. 首批指令样本构建

### 9.1 分层抽取业务记录

从已划分的数据集合中分别抽取：
- Training：80 条业务记录。
- Validation：10 条业务记录。
- Test：10 条业务记录。

每条记录后续生成 Task A、Task B 各一条，
预计共生成 200 条指令样本。

抽样固定随机种子，尽量保持原标签比例。
首批数据用于验证流程，小规模测试结果不作为正式性能结论。

In [ ]:
# 9.1.1 从各集合中分层抽样
PILOT_SIZES = {
    "train": 80,
    "validation": 10,
    "test": 10,
}

pilot_records = {}

for name, df in record_splits.items():
    selected, _ = train_test_split(
        df,
        train_size=PILOT_SIZES[name],
        random_state=SEED,
        stratify=df["delay_label"],
    )

    pilot_records[name] = selected.copy().reset_index(drop=True)

# 9.1.2 验证归属与唯一性
for name, df in pilot_records.items():
    assert len(df) == PILOT_SIZES[name]
    assert df["load_id"].is_unique
    assert set(df["load_id"]).issubset(
        set(record_splits[name]["load_id"])
    )

all_pilot_ids = [
    load_id
    for df in pilot_records.values()
    for load_id in df["load_id"]
]
assert len(all_pilot_ids) == len(set(all_pilot_ids))

# 9.1.3 汇总实际抽样数量
pilot_summary = pd.DataFrame([
    {
        "split": name,
        "business_records": len(df),
        "delayed": int(df["delay_label"].eq("delayed").sum()),
        "on_time": int(df["delay_label"].eq("on_time").sum()),
        "planned_instruction_samples": len(df) * 2,
    }
    for name, df in pilot_records.items()
])

display(pilot_summary)
print("业务记录总数:", sum(len(df) for df in pilot_records.values()))
print("检查通过：抽样归属正确，运单无重复")

### 9.2 转换为 Chat Messages 格式

每条业务记录生成两条指令：
- Task A：提取八个业务字段，输出 JSON。
- Task B：根据计划与实际交付时间判断是否延误，输出 JSON。

保留来源运单、运输任务编号和数据集合归属，便于追溯。
模型输入不包含派生时间差、准时标记或答案标签。

首批使用固定文本模板验证流程。
正式扩展时增加表达多样性和缺失信息场景。

In [ ]:
import json

# 9.2.1 统一输出要求
SYSTEM_PROMPT = (
    "你是物流业务数据助手。"
    "只依据用户提供的记录完成任务，不编造信息。"
    "只输出合法 JSON，不添加解释或 Markdown 代码围栏。"
)

A_FIELDS = [
    "load_id",
    "truck_id",
    "origin_city",
    "destination_city",
    "revenue_usd",
    "fuel_used_gallons",
    "scheduled_delivery",
    "actual_delivery",
]


# 9.2.2 格式化函数
def timestamp_text(value):
    # 保留微秒，避免舍入改变时间先后关系
    return pd.Timestamp(value).isoformat(sep=" ", timespec="microseconds")


def json_text(value):
    return json.dumps(value, ensure_ascii=False, allow_nan=False)


def make_sample(row, split, task, user_text, answer):
    return {
        "sample_id": f"{row['load_id']}_{task}",
        "source_load_id": str(row["load_id"]),
        "source_trip_id": str(row["trip_id"]),
        "split": split,
        "task": task,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": json_text(answer)},
        ],
    }


# 9.2.3 每条记录生成 Task A、Task B 各一条
instruction_splits = {}

for split, df in pilot_records.items():
    samples = []

    for _, row in df.iterrows():
        scheduled = timestamp_text(row["delivery_scheduled_datetime"])
        actual = timestamp_text(row["delivery_actual_datetime"])
        truck_id = None if pd.isna(row["truck_id"]) else str(row["truck_id"])

        # Task A：信息提取
        answer_a = {
            "load_id": str(row["load_id"]),
            "truck_id": truck_id,
            "origin_city": str(row["origin_city"]),
            "destination_city": str(row["destination_city"]),
            "revenue_usd": float(row["revenue"]),
            "fuel_used_gallons": float(row["fuel_gallons_used"]),
            "scheduled_delivery": scheduled,
            "actual_delivery": actual,
        }

        user_a = (
            f"从记录中提取以下字段：{', '.join(A_FIELDS)}。"
            "编号、城市和时间使用字符串；收入与耗油量使用数字。"
            "时间保留原格式，缺失信息填 null。\n\n"
            f"运单编号：{answer_a['load_id']}；"
            f"车辆编号：{truck_id if truck_id is not None else '未提供'}；"
            f"出发城市：{answer_a['origin_city']}；"
            f"目的城市：{answer_a['destination_city']}；"
            f"运单收入：{answer_a['revenue_usd']} 美元；"
            f"实际耗油量：{answer_a['fuel_used_gallons']} 加仑；"
            f"计划交付时间：{scheduled}；"
            f"实际交付时间：{actual}。"
        )

        samples.append(
            make_sample(row, split, "A", user_a, answer_a)
        )

        # Task B：只提供时间和判断规则，不提供标签或时间差
        user_b = (
            "判断以下记录是否发生交付延误。"
            "所有时间使用同一时间口径。"
            "实际交付时间晚于计划时间标为 delayed，"
            "否则标为 on_time。"
            '输出格式为 {"delay_label": "标签"}。\n\n'
            f"计划交付时间：{scheduled}；"
            f"实际交付时间：{actual}。"
        )

        samples.append(
            make_sample(
                row, split, "B", user_b,
                {"delay_label": str(row["delay_label"])},
            )
        )

    instruction_splits[split] = samples


# 9.2.4 汇总数量
display(pd.DataFrame([
    {
        "split": split,
        "task_A": sum(s["task"] == "A" for s in samples),
        "task_B": sum(s["task"] == "B" for s in samples),
        "total": len(samples),
    }
    for split, samples in instruction_splits.items()
]))

# 只展示训练集的第一组样本
for sample in instruction_splits["train"][:2]:
    print(json.dumps(sample, ensure_ascii=False, indent=2))
    print()

### 9.3 样本格式与答案一致性检查

检查：
1. 样本编号唯一，来源与集合归属正确。
2. messages 包含 system、user、assistant 三个角色。
3. 标准答案可以解析为 JSON，字段及类型符合要求。
4. Task A 答案与来源业务记录一致。
5. Task B 答案与计划、实际交付时间的比较结果一致。
6. 不同集合之间没有完全相同的模型输入。

In [ ]:
# 9.3.1 准备来源索引与检查容器
source_lookup = clean_records.set_index("load_id")

SOURCE_FIELDS = {
    "load_id": "load_id",
    "truck_id": "truck_id",
    "origin_city": "origin_city",
    "destination_city": "destination_city",
    "revenue_usd": "revenue",
    "fuel_used_gallons": "fuel_gallons_used",
    "scheduled_delivery": "delivery_scheduled_datetime",
    "actual_delivery": "delivery_actual_datetime",
}

NUMERIC_FIELDS = {"revenue_usd", "fuel_used_gallons"}
TIME_FIELDS = {"scheduled_delivery", "actual_delivery"}

errors = []
seen_ids = set()
input_sets = {name: set() for name in instruction_splits}
validation_summary = []

# 9.3.2 逐条检查，发现问题时记录而不修改样本
for split, samples in instruction_splits.items():
    passed = 0
    allowed_ids = set(record_splits[split]["load_id"])

    for sample in samples:
        sample_id = sample.get("sample_id", "unknown")

        try:
            assert sample_id not in seen_ids, "样本编号重复"
            seen_ids.add(sample_id)

            load_id = sample["source_load_id"]
            assert sample["split"] == split, "集合标记错误"
            assert load_id in allowed_ids, "来源运单不属于当前集合"

            row = source_lookup.loc[load_id]
            assert sample["source_trip_id"] == row["trip_id"], "运输编号不一致"

            messages = sample["messages"]
            assert [m["role"] for m in messages] == [
                "system", "user", "assistant"
            ], "角色或顺序错误"
            assert all(
                isinstance(m["content"], str) and m["content"].strip()
                for m in messages
            ), "消息内容为空或不是字符串"

            input_sets[split].add((
                messages[0]["content"],
                messages[1]["content"],
            ))

            answer = json.loads(messages[2]["content"])
            assert isinstance(answer, dict), "答案不是 JSON 对象"

            if sample["task"] == "A":
                assert set(answer) == set(SOURCE_FIELDS), "Task A 字段不符"

                for field, source_field in SOURCE_FIELDS.items():
                    expected = (
                        load_id if source_field == "load_id"
                        else row[source_field]
                    )
                    value = answer[field]

                    if pd.isna(expected):
                        assert value is None, f"{field} 应为 null"
                    elif field in NUMERIC_FIELDS:
                        assert type(value) in (int, float), f"{field} 类型错误"
                        assert np.isfinite(value), f"{field} 不是有限数值"
                        assert value == float(expected), f"{field} 数值不符"
                    elif field in TIME_FIELDS:
                        assert isinstance(value, str), f"{field} 类型错误"
                        assert pd.Timestamp(value) == expected, f"{field} 时间不符"
                    else:
                        assert isinstance(value, str), f"{field} 类型错误"
                        assert value == str(expected), f"{field} 内容不符"

            elif sample["task"] == "B":
                assert set(answer) == {"delay_label"}, "Task B 字段不符"
                expected_label = (
                    "delayed"
                    if row["delivery_actual_datetime"]
                    > row["delivery_scheduled_datetime"]
                    else "on_time"
                )
                assert answer["delay_label"] == expected_label, "延误标签错误"

            else:
                raise ValueError("未知任务类型")

            passed += 1

        except (AssertionError, ValueError, TypeError, KeyError, IndexError) as exc:
            errors.append({
                "split": split,
                "sample_id": sample_id,
                "error": str(exc),
            })

    validation_summary.append({
        "split": split,
        "checked": len(samples),
        "passed": passed,
        "failed": len(samples) - passed,
    })

# 9.3.3 汇总并检查跨集合输入重复
display(pd.DataFrame(validation_summary))

overlap_summary = pd.DataFrame([
    {
        "pair": f"{left} / {right}",
        "identical_inputs": len(input_sets[left] & input_sets[right]),
    }
    for left, right in [
        ("train", "validation"),
        ("train", "test"),
        ("validation", "test"),
    ]
])

display(overlap_summary)

if errors:
    display(pd.DataFrame(errors))
else:
    print("全部样本的格式与来源答案检查通过")

print("注意：本检查不替代后续人工抽查与表达多样性评估。")

## 10. 导出首批数据集

导出流程验证版：
- train.jsonl：160 条。
- val.jsonl：20 条。
- test.jsonl：20 条。

每行保存一条完整样本，包含 messages 和来源信息。
后续训练仅使用 messages，来源信息用于追溯。

同时保存：
- split_assignments.csv：全部业务记录的集合归属。
- dataset_info.json：数据规模、随机种子和任务规则。

写入后重新读取验证，最后打包下载。
本版本用于流程验证，不作为正式性能评测数据集。

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

# 10.2.1 确认上一阶段检查通过
assert not errors, "存在未解决的样本错误"
assert overlap_summary["identical_inputs"].eq(0).all(), "存在跨集合重复输入"

output_dir = Path("fleetmind_phase1_pilot")
output_dir.mkdir(exist_ok=True)

FILE_MAP = {
    "train": "train.jsonl",
    "validation": "val.jsonl",
    "test": "test.jsonl",
}

exported_files = []
export_summary = []

# 10.2.2 写入 JSONL，并重新读取核对
for split, filename in FILE_MAP.items():
    samples = instruction_splits[split]
    path = output_dir / filename

    with path.open("w", encoding="utf-8") as f:
        for sample in samples:
            f.write(json.dumps(
                sample, ensure_ascii=False, allow_nan=False
            ) + "\n")

    with path.open("r", encoding="utf-8") as f:
        restored = [json.loads(line) for line in f if line.strip()]

    assert restored == samples, f"{filename} 回读不一致"

    exported_files.append(path)
    export_summary.append({
        "file": filename,
        "samples": len(restored),
        "size_KB": round(path.stat().st_size / 1024, 2),
    })

# 10.2.3 保存全部业务记录的划分，供后续扩展使用
split_assignments = pd.concat([
    df[["load_id", "trip_id"]].assign(split=name)
    for name, df in record_splits.items()
], ignore_index=True)

assert split_assignments["load_id"].is_unique
assert len(split_assignments) == len(task_records)

assignment_path = output_dir / "split_assignments.csv"
split_assignments.to_csv(assignment_path, index=False)
exported_files.append(assignment_path)

# 10.2.4 保存数据说明
dataset_info = {
    "project": "FleetMind-LLM",
    "version": "pilot",
    "source": "yogape/logistics-operations",
    "source_type": "synthetic",
    "seed": SEED,
    "clean_business_records": len(clean_records),
    "pilot_business_records": sum(
        len(df) for df in pilot_records.values()
    ),
    "instruction_counts": {
        name: len(samples)
        for name, samples in instruction_splits.items()
    },
    "task_A_fields": A_FIELDS,
    "task_B_rule": {
        "delayed": "actual_delivery > scheduled_delivery",
        "on_time": "actual_delivery <= scheduled_delivery",
    },
    "currency": "USD",
    "fuel_unit": "gallons",
    "note": "固定模板流程验证版，不用于正式性能结论",
}

info_path = output_dir / "dataset_info.json"
info_path.write_text(
    json.dumps(dataset_info, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
exported_files.append(info_path)

# 10.2.5 打包并下载
zip_path = Path("FleetMind_LLM_Phase1_Pilot.zip")

with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
    for path in exported_files:
        archive.write(path, arcname=f"{output_dir.name}/{path.name}")

display(pd.DataFrame(export_summary))
print("JSONL 回读验证通过")
print("完整业务划分已保存:", len(split_assignments), "条")
print("压缩包:", zip_path.name)

files.download(str(zip_path))

## 11. 人工抽查训练样本

抽查：
1. 缺失车辆编号是否正确输出 null（若首批训练样本中存在）。
2. 普通信息提取样本是否清楚、单位是否正确。
3. 未延误分类样本的时间关系与答案是否一致。

仅检查训练集，测试集继续保留。

In [ ]:
import random

rng = random.Random(SEED)
train_samples = instruction_splits["train"]

task_a_samples = [s for s in train_samples if s["task"] == "A"]
missing_truck_samples = [
    s for s in task_a_samples
    if json.loads(s["messages"][-1]["content"])["truck_id"] is None
]
on_time_samples = [
    s for s in train_samples
    if s["task"] == "B"
    and json.loads(s["messages"][-1]["content"])["delay_label"] == "on_time"
]

review_samples = []

# 11.1 优先查看一条缺失车辆编号的样本
if missing_truck_samples:
    review_samples.append(rng.choice(missing_truck_samples))
else:
    print("首批训练集没有车辆编号缺失样本，正式扩展时需补充。")

# 11.2 查看另一条信息提取样本
selected_ids = {s["sample_id"] for s in review_samples}
remaining_a = [
    s for s in task_a_samples if s["sample_id"] not in selected_ids
]
review_samples.append(rng.choice(remaining_a))

# 11.3 查看一条未延误样本
review_samples.append(rng.choice(on_time_samples))

for index, sample in enumerate(review_samples, start=1):
    print(f"\n样本 {index}：{sample['sample_id']}")
    print("输入：")
    print(sample["messages"][1]["content"])
    print("\n标准答案：")
    print(sample["messages"][2]["content"])

## 12. 正式版数据扩展

### 12.1 扩展业务记录抽样

在已有划分内扩展至：
Training 800 条、Validation 100 条、Test 100 条业务记录。

保留首批已抽取的记录，再从同一集合补充抽样。
固定随机种子，新增记录按延误标签分层抽样。

首批数据保持不变，正式版使用独立变量保存。
样本表达多样性与缺失信息场景在后续步骤补充。

In [ ]:
# 12.1.1 设置正式版业务记录数
FORMAL_SIZES = {
    "train": 800,
    "validation": 100,
    "test": 100,
}

formal_record_splits = {}

for name, target_size in FORMAL_SIZES.items():
    existing = pilot_records[name].copy()

    # 从同一集合中排除已经选过的运单
    remaining = record_splits[name].loc[
        ~record_splits[name]["load_id"].isin(existing["load_id"])
    ]

    extra, _ = train_test_split(
        remaining,
        train_size=target_size - len(existing),
        random_state=SEED,
        stratify=remaining["delay_label"],
    )

    combined = pd.concat([existing, extra], ignore_index=True)
    combined = combined.sample(
        frac=1, random_state=SEED
    ).reset_index(drop=True)

    assert len(combined) == target_size
    assert combined["load_id"].is_unique
    assert set(existing["load_id"]).issubset(set(combined["load_id"]))
    assert set(combined["load_id"]).issubset(
        set(record_splits[name]["load_id"])
    )

    formal_record_splits[name] = combined

# 12.1.2 汇总并检查跨集合唯一性
all_ids = [
    load_id
    for df in formal_record_splits.values()
    for load_id in df["load_id"]
]
assert len(all_ids) == len(set(all_ids))

display(pd.DataFrame([
    {
        "split": name,
        "business_records": len(df),
        "delayed": int(df["delay_label"].eq("delayed").sum()),
        "on_time": int(df["delay_label"].eq("on_time").sum()),
        "planned_instructions": len(df) * 2,
    }
    for name, df in formal_record_splits.items()
]))

print("检查通过：首批记录已保留，集合归属不变，运单无重复")

### 12.2 表达变化与缺失信息规则

Task A：
- 使用不同业务表达，并打乱字段顺序。
- 25% 的样本主动省略 1–2 个字段。
- 原始缺失或输入中省略的信息，答案均为 null。
- 保留运单编号，不编造地点、金额、单位或异常原因。

Task B：
- 保留完整的计划和实际交付时间。
- 使用不同表达，但保持相同标签规则。

不同集合使用不同句式模板。
模板变化用于初步检验表达适应能力，
不能替代真实业务文本上的泛化评测。

In [ ]:
# 12.2.1 配置输入中可以主动省略的字段
import random

OMIT_RATE = 0.25
OMITTABLE_FIELDS = [
    field for field in A_FIELDS if field != "load_id"
]

# Task A：不同集合使用不同句式
A_TEMPLATES = {
    "train": [
        "运输记录如下：{details}",
        "本趟业务登记：{details}",
        "请处理以下运输交接信息：{details}",
        "收到一份运输业务摘要：{details}",
    ],
    "validation": [
        "以下是待录入系统的运输资料：{details}",
        "请整理这份业务登记内容：{details}",
    ],
    "test": [
        "运营人员提供了如下运输说明：{details}",
        "请从这份运输情况记录中读取信息：{details}",
    ],
}

# Task B：仍然只输入计划与实际交付时间
B_TEMPLATES = {
    "train": [
        "计划交付时间：{scheduled}；实际交付时间：{actual}。",
        "这趟运输原定于{scheduled}交付，实际于{actual}交付。",
        "交付登记：实际时间为{actual}，计划时间为{scheduled}。",
        "请核对交付记录：计划{scheduled}，实际{actual}。",
    ],
    "validation": [
        "交付记录列明：原计划{scheduled}；实际完成于{actual}。",
        "请比较本次交付时间：实际为{actual}；约定计划为{scheduled}。",
    ],
    "test": [
        "运营记录记载，计划交付为{scheduled}，实际交付为{actual}。",
        "本次交付已完成于{actual}，对应计划交付时刻为{scheduled}。",
    ],
}

# 12.2.2 为每条业务记录固定省略字段
# 使用独立随机数生成器，重复运行可得到同一方案
omission_plans = {}
plan_summary = []

for index, (split, df) in enumerate(formal_record_splits.items()):
    rng = random.Random(SEED + index)
    load_ids = df["load_id"].tolist()

    selected_ids = set(
        rng.sample(load_ids, k=round(len(load_ids) * OMIT_RATE))
    )

    omission_plans[split] = {
        load_id: (
            rng.sample(OMITTABLE_FIELDS, k=rng.choice([1, 2]))
            if load_id in selected_ids else []
        )
        for load_id in load_ids
    }

    plan_summary.append({
        "split": split,
        "business_records": len(df),
        "A_with_planned_omissions": len(selected_ids),
        "A_templates": len(A_TEMPLATES[split]),
        "B_templates": len(B_TEMPLATES[split]),
    })

display(pd.DataFrame(plan_summary))
print("生成规则已固定，尚未修改或生成正式版指令")

### 12.3 生成正式版指令样本

按已固定的模板和省略方案生成样本：
- Task A：打乱字段顺序，省略的信息对应答案为 null。
- Task B：根据完整交付时间生成分类答案。

保留模板编号、省略字段和实际展示字段，供后续审核。
这些信息属于追溯元数据，不作为模型输入。

In [ ]:
# 12.3.1 中文字段名称
FIELD_LABELS = {
    "load_id": "运单编号",
    "truck_id": "车辆编号",
    "origin_city": "出发城市",
    "destination_city": "目的城市",
    "revenue_usd": "运单收入（美元）",
    "fuel_used_gallons": "实际耗油量（加仑）",
    "scheduled_delivery": "计划交付时间",
    "actual_delivery": "实际交付时间",
}

A_INSTRUCTION = (
    f"提取以下字段：{', '.join(A_FIELDS)}。"
    "编号、城市和时间使用字符串，收入与耗油量使用数字。"
    "时间保留原格式。输入未提供的信息填 null，不要推测。\n\n"
)

B_INSTRUCTION = (
    "判断记录是否发生交付延误。所有时间使用同一时间口径。"
    "实际交付时间晚于计划时间标为 delayed，否则标为 on_time。"
    '只输出 {"delay_label": "标签"}。\n\n'
)

formal_instruction_splits = {}

# 12.3.2 按集合分别生成，每条记录对应 A、B 两个任务
for split_index, (split, df) in enumerate(formal_record_splits.items()):
    rng = random.Random(SEED + 100 + split_index)
    samples = []

    for position, (_, row) in enumerate(df.iterrows()):
        scheduled = timestamp_text(row["delivery_scheduled_datetime"])
        actual = timestamp_text(row["delivery_actual_datetime"])

        source_values = {
            "load_id": str(row["load_id"]),
            "truck_id": (
                None if pd.isna(row["truck_id"]) else str(row["truck_id"])
            ),
            "origin_city": str(row["origin_city"]),
            "destination_city": str(row["destination_city"]),
            "revenue_usd": float(row["revenue"]),
            "fuel_used_gallons": float(row["fuel_gallons_used"]),
            "scheduled_delivery": scheduled,
            "actual_delivery": actual,
        }

        # Task A：先决定展示哪些字段，再据此构建答案
        omitted_fields = omission_plans[split][str(row["load_id"])]

        shown_fields = [
            field for field in A_FIELDS
            if field not in omitted_fields
            and source_values[field] is not None
        ]
        rng.shuffle(shown_fields)

        details = "；".join(
            f"{FIELD_LABELS[field]}：{source_values[field]}"
            for field in shown_fields
        ) + "。"

        answer_a = {
            field: (
                source_values[field] if field in shown_fields else None
            )
            for field in A_FIELDS
        }

        a_index = position % len(A_TEMPLATES[split])
        user_a = A_INSTRUCTION + A_TEMPLATES[split][a_index].format(
            details=details
        )

        sample_a = make_sample(row, split, "A", user_a, answer_a)
        sample_a.update({
            "template_id": f"{split}_A_{a_index + 1}",
            "omitted_fields": list(omitted_fields),
            "shown_fields": list(shown_fields),
        })
        samples.append(sample_a)

        # Task B：从时间重新计算答案，不依赖展示中的标签
        b_index = position % len(B_TEMPLATES[split])
        user_b = B_INSTRUCTION + B_TEMPLATES[split][b_index].format(
            scheduled=scheduled,
            actual=actual,
        )

        delay_label = (
            "delayed"
            if row["delivery_actual_datetime"]
            > row["delivery_scheduled_datetime"]
            else "on_time"
        )

        sample_b = make_sample(
            row, split, "B", user_b,
            {"delay_label": delay_label},
        )
        sample_b["template_id"] = f"{split}_B_{b_index + 1}"
        samples.append(sample_b)

    formal_instruction_splits[split] = samples

# 12.3.3 汇总数量
formal_summary = pd.DataFrame([
    {
        "split": split,
        "task_A": sum(s["task"] == "A" for s in samples),
        "task_B": sum(s["task"] == "B" for s in samples),
        "A_with_planned_omissions": sum(
            s["task"] == "A" and bool(s["omitted_fields"])
            for s in samples
        ),
        "total": len(samples),
    }
    for split, samples in formal_instruction_splits.items()
])

display(formal_summary)

# 12.3.4 只展示训练集：一条含省略字段的 A、一条 B
train_formal = formal_instruction_splits["train"]

example_a = next(
    s for s in train_formal
    if s["task"] == "A" and s["omitted_fields"]
)
example_b = next(s for s in train_formal if s["task"] == "B")

for sample in [example_a, example_b]:
    print(json.dumps(sample, ensure_ascii=False, indent=2))
    print()

### 12.4 正式版样本检查

检查消息结构、样本唯一性、来源及集合归属。
Task A 根据输入中实际出现的字段审核：
- 已展示字段与来源记录一致。
- 未展示字段的答案为 null。
- 输入字段与 shown_fields、省略方案一致。

Task B 根据来源时间独立计算标签，并检查输入包含对应时间。
检查每个运单各有一条 A、B 样本，以及跨集合的输入重复情况。

In [ ]:
import re
from collections import Counter

# 12.4.1 使用独立检查变量，不覆盖 Pilot 检查结果
formal_errors = []
formal_seen_ids = set()
formal_input_sets = {name: set() for name in formal_instruction_splits}
formal_source_sets = {name: set() for name in formal_instruction_splits}
formal_check_rows = []

def reject_json_constant(value):
    raise ValueError(f"JSON 包含非法常量：{value}")

for split, samples in formal_instruction_splits.items():
    allowed_ids = set(formal_record_splits[split]["load_id"])
    task_counts = Counter()
    passed = 0

    for sample in samples:
        sample_id = sample.get("sample_id", "unknown")

        try:
            assert sample_id not in formal_seen_ids, "样本编号重复"
            formal_seen_ids.add(sample_id)

            load_id = sample["source_load_id"]
            task = sample["task"]
            assert task in ("A", "B"), "未知任务"
            assert load_id in allowed_ids, "来源或集合归属错误"
            assert sample["split"] == split, "集合标记错误"

            row = source_lookup.loc[load_id]
            assert sample["source_trip_id"] == row["trip_id"], "运输编号不一致"

            messages = sample["messages"]
            assert [m["role"] for m in messages] == [
                "system", "user", "assistant"
            ], "消息角色或顺序错误"
            assert all(
                isinstance(m["content"], str) and m["content"].strip()
                for m in messages
            ), "消息内容无效"

            user_text = messages[1]["content"]
            answer = json.loads(
                messages[2]["content"],
                parse_constant=reject_json_constant,
            )
            assert isinstance(answer, dict), "答案不是 JSON 对象"

            if task == "A":
                assert set(answer) == set(SOURCE_FIELDS), "输出字段不符"

                # 从实际输入解析字段，不只依赖 shown_fields 元数据
                extracted = {}
                for field, label in FIELD_LABELS.items():
                    matches = re.findall(
                        re.escape(label) + r"：([^；。\n]+)",
                        user_text,
                    )
                    assert len(matches) <= 1, f"{field} 在输入中重复"
                    if matches:
                        extracted[field] = matches[0].strip()

                shown = sample["shown_fields"]
                assert len(shown) == len(set(shown)), "展示字段列表重复"
                assert set(extracted) == set(shown), "输入与展示字段记录不一致"

                omitted = sample["omitted_fields"]
                assert omitted == omission_plans[split][load_id], "省略方案不一致"

                expected_shown = set()
                for field, source_field in SOURCE_FIELDS.items():
                    original = (
                        load_id if source_field == "load_id"
                        else row[source_field]
                    )

                    should_show = field not in omitted and not pd.isna(original)

                    if not should_show:
                        assert field not in extracted, f"{field} 不应出现在输入"
                        assert answer[field] is None, f"{field} 应为 null"
                        continue

                    expected_shown.add(field)
                    assert field in extracted, f"{field} 应展示但未展示"
                    value = answer[field]
                    text_value = extracted[field]

                    if field in NUMERIC_FIELDS:
                        assert type(value) in (int, float), f"{field} 类型错误"
                        assert np.isfinite(value), f"{field} 数值无效"
                        assert float(text_value) == float(original), f"{field} 输入错误"
                        assert value == float(original), f"{field} 答案错误"
                    elif field in TIME_FIELDS:
                        assert isinstance(value, str), f"{field} 类型错误"
                        assert pd.Timestamp(text_value) == pd.Timestamp(original), f"{field} 输入错误"
                        assert value == text_value, f"{field} 未保留输入时间格式"
                    else:
                        assert isinstance(value, str), f"{field} 类型错误"
                        assert text_value == str(original), f"{field} 输入错误"
                        assert value == text_value, f"{field} 答案错误"

                assert set(extracted) == expected_shown, "存在额外或缺失字段"

            else:
                assert set(answer) == {"delay_label"}, "分类输出字段不符"

                scheduled = row["delivery_scheduled_datetime"]
                actual = row["delivery_actual_datetime"]

                # 依据来源时间独立计算，不读取已有 delay_label
                expected_label = "delayed" if actual > scheduled else "on_time"
                assert answer["delay_label"] == expected_label, "延误标签错误"
                assert timestamp_text(scheduled) in user_text, "输入计划时间不符"
                assert timestamp_text(actual) in user_text, "输入实际时间不符"

            formal_input_sets[split].add((
                messages[0]["content"], user_text
            ))
            formal_source_sets[split].add(load_id)
            task_counts[(load_id, task)] += 1
            passed += 1

        except (AssertionError, ValueError, TypeError, KeyError, IndexError) as exc:
            formal_errors.append({
                "split": split,
                "sample_id": sample_id,
                "error": str(exc),
            })

    pair_issues = sum(
        task_counts[(load_id, "A")] != 1
        or task_counts[(load_id, "B")] != 1
        for load_id in allowed_ids
    )

    formal_check_rows.append({
        "split": split,
        "checked": len(samples),
        "passed": passed,
        "failed": len(samples) - passed,
        "AB_pair_issues": pair_issues,
    })

# 12.4.2 汇总跨集合检查
formal_check_summary = pd.DataFrame(formal_check_rows)

formal_overlap_summary = pd.DataFrame([
    {
        "pair": f"{left} / {right}",
        "shared_loads": len(
            formal_source_sets[left] & formal_source_sets[right]
        ),
        "identical_inputs": len(
            formal_input_sets[left] & formal_input_sets[right]
        ),
    }
    for left, right in [
        ("train", "validation"),
        ("train", "test"),
        ("validation", "test"),
    ]
])

display(formal_check_summary)
display(formal_overlap_summary)

if formal_errors:
    display(pd.DataFrame(formal_errors).head(20))
else:
    print("正式版样本格式、输入与答案一致性检查通过")

### 12.5 导出正式版数据

导出：
- train.jsonl：1,600 条。
- val.jsonl：200 条。
- test.jsonl：200 条。
- split_assignments.csv：全部可用业务记录的集合归属。
- dataset_info.json：数据规模、任务规则及生成配置。

每条样本保留 messages 和追溯元数据。
后续训练仅使用 messages。

本版本覆盖信息提取和已发生交付记录的延误分类，
基于合成业务数据及模板构建，不代表真实业务泛化效果。

In [ ]:
# 12.5.1 导出前检查
assert not formal_errors, "仍存在样本错误"
assert formal_check_summary[
    ["failed", "AB_pair_issues"]
].eq(0).all().all()

assert formal_overlap_summary[
    ["shared_loads", "identical_inputs"]
].eq(0).all().all()

formal_dir = Path("fleetmind_phase1_v1")
formal_dir.mkdir(exist_ok=True)

formal_files = []
formal_export_rows = []

# 12.5.2 写入 JSONL，重新读取核对
for split, filename in FILE_MAP.items():
    samples = formal_instruction_splits[split]
    assert len(samples) == FORMAL_SIZES[split] * 2

    path = formal_dir / filename

    with path.open("w", encoding="utf-8") as f:
        for sample in samples:
            f.write(json.dumps(
                sample, ensure_ascii=False, allow_nan=False
            ) + "\n")

    with path.open("r", encoding="utf-8") as f:
        restored = [json.loads(line) for line in f if line.strip()]

    assert restored == samples, f"{filename} 回读不一致"

    formal_files.append(path)
    formal_export_rows.append({
        "file": filename,
        "samples": len(restored),
        "size_KB": round(path.stat().st_size / 1024, 2),
    })

# 12.5.3 保存全部业务记录的划分
assignment_path = formal_dir / "split_assignments.csv"
split_assignments.to_csv(assignment_path, index=False)
formal_files.append(assignment_path)

# 12.5.4 保存数据说明与生成配置
formal_info = {
    "project": "FleetMind-LLM",
    "version": "v1",
    "source": "https://huggingface.co/datasets/yogape/logistics-operations",
    "source_type": "synthetic",
    "seed": SEED,
    "clean_business_records": len(clean_records),
    "selected_business_records": {
        name: len(df)
        for name, df in formal_record_splits.items()
    },
    "instruction_counts": {
        name: len(samples)
        for name, samples in formal_instruction_splits.items()
    },
    "task_A_fields": A_FIELDS,
    "task_A_omission_rate": OMIT_RATE,
    "task_A_missing_value": None,
    "task_B_rule": {
        "delayed": "actual_delivery > scheduled_delivery",
        "on_time": "actual_delivery <= scheduled_delivery",
    },
    "currency": "USD",
    "fuel_unit": "gallons",
    "system_prompt": SYSTEM_PROMPT,
    "task_A_instruction": A_INSTRUCTION,
    "task_B_instruction": B_INSTRUCTION,
    "task_A_templates": A_TEMPLATES,
    "task_B_templates": B_TEMPLATES,
    "omission_plans": omission_plans,
    "validation_summary": formal_check_summary.to_dict("records"),
    "overlap_summary": formal_overlap_summary.to_dict("records"),
    "scope": "物流信息提取及已发生交付记录的延误分类",
    "limitations": [
        "来源为合成数据，文本由模板构建",
        "尚未覆盖费用风险和完整经营推理",
        "尚未使用独立真实业务文本验证泛化",
    ],
}

info_path = formal_dir / "dataset_info.json"
info_path.write_text(
    json.dumps(
        formal_info, ensure_ascii=False, indent=2, allow_nan=False
    ),
    encoding="utf-8",
)
formal_files.append(info_path)

# 12.5.5 打包并下载
formal_zip = Path("FleetMind_LLM_Phase1_Dataset.zip")

with ZipFile(formal_zip, "w", compression=ZIP_DEFLATED) as archive:
    for path in formal_files:
        archive.write(path, arcname=f"{formal_dir.name}/{path.name}")

display(pd.DataFrame(formal_export_rows))
print("正式版 JSONL 回读验证通过")
print("总指令数:", sum(
    len(samples) for samples in formal_instruction_splits.values()
))
print("压缩包:", formal_zip.name)

files.download(str(formal_zip))

## 13. Phase 1 阶段总结

### 13.1 数据来源与处理

采用 Logistics Operations Database 合成物流运营数据。

读取并关联四张表：
loads、trips、routes、delivery_events。

- 合并后业务记录：85,410 条。
- 排除时间顺序异常：634 条。
- 保留业务记录：84,776 条，保留率 99.26%。
- 保留司机、车辆和挂车编号缺失的记录。
- 主键、重复、关联及已执行的业务逻辑检查完成。

### 13.2 任务与样本

Task A：物流信息提取，输出八个字段的 JSON。
Task B：根据计划与实际交付时间，判断已发生的交付是否延误。

标签规则：
- delayed：实际交付时间晚于计划时间。
- on_time：实际交付时间不晚于计划时间。

选取 1,000 条业务记录，每条生成 A、B 各一条指令，
共 2,000 条 Chat Messages 样本。

| 集合 | 业务记录 | 指令样本 |
|---|---:|---:|
| Training | 800 | 1,600 |
| Validation | 100 | 200 |
| Test | 100 | 200 |

### 13.3 多样性与检查

- 固定随机种子为 42。
- 按运单划分，延误标签采用分层抽样。
- 不同集合使用不同句式模板。
- 打乱 Task A 的输入字段顺序。
- 25% 的 Task A 样本主动省略 1–2 个字段。
- 输入未提供的信息，标准答案为 null。
- 2,000 条样本全部通过已执行的自动检查。
- A/B 配对完整，集合间运单无重叠。
- 未发现跨集合完全相同的模型输入。
- JSONL 写入后回读一致。

### 13.4 产出与限制

正式版压缩包：
FleetMind_LLM_Phase1_Dataset.zip

包含 train.jsonl、val.jsonl、test.jsonl、
split_assignments.csv 和 dataset_info.json。

当前数据基于合成业务记录及模板构建，
尚未覆盖费用风险、完整经营推理或真实业务文本泛化评测。
Task B 属于已发生记录的延误识别，不是未来延误预测。

### 13.5 下一阶段

Phase 2：使用未进行本项目微调的 Qwen3-0.6B 建立基线。

先在验证集上检查推理及评分流程，固定评测设置。
测试集保留到最终对比阶段，不用于调整提示词或参数。